# Combinatorics and Counting

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.04 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/07_Discrete_Mathematics/01_combinatorics_and_counting.ipynb)

---

## 🎯 Learning Objective

Learn permutations, combinations, and the binomial and multinomial coefficients that count discrete outcomes.

---

## 📐 Theory

Discrete mathematics starts from a simple question: *how many ways can something happen?*
That sounds trivial for small numbers, but the answer grows so fast that counting carefully —
rather than one at a time — becomes the only option. Every algorithm in this notebook is a
technique for counting without listing.

### The multiplication principle

If a process has $k$ independent stages, and stage $i$ has $n_i$ possible choices, the total
number of outcomes is $n_1 \times n_2 \times \cdots \times n_k$. This is the **multiplication
principle**, and it's the seed every other counting formula grows from: choosing an outfit from
3 shirts and 4 pants gives $3\times 4=12$ outfits, because each shirt choice is independent of
each pants choice.

### Permutations: order matters

A **permutation** counts arrangements where order matters. The number of ways to arrange all
$n$ distinct items in a sequence is $n! = n\times(n-1)\times\cdots\times 1$ — applying the
multiplication principle to "$n$ choices for slot 1, $n-1$ remaining choices for slot 2, ..."
The number of ways to arrange just $k$ of $n$ items (a **k-permutation**) is

$$P(n,k) = \frac{n!}{(n-k)!} = n\times(n-1)\times\cdots\times(n-k+1),$$

again the multiplication principle, stopped after $k$ factors instead of $n$.

### Combinations: order doesn't matter

A **combination** counts *subsets* of size $k$ — order thrown away. Since every subset of
size $k$ can be arranged in $k!$ orders, and $P(n,k)$ counts every subset once per arrangement,

$$\binom{n}{k} = \frac{P(n,k)}{k!} = \frac{n!}{k!(n-k)!}$$

is the **binomial coefficient**, read "n choose k." It's symmetric ($\binom{n}{k}=\binom{n}{n-k}$,
choosing which $k$ to keep is the same act as choosing which $n-k$ to discard) and satisfies
**Pascal's identity** $\binom{n}{k}=\binom{n-1}{k-1}+\binom{n-1}{k}$ — either item $n$ is in
your chosen subset (then choose $k-1$ more from the remaining $n-1$) or it isn't (then choose
all $k$ from the remaining $n-1$). This recursive splitting is exactly what generates Pascal's
triangle, and it's also the same "split on one variable, recurse on the rest" pattern that
dynamic programming (`07.03`, [Trees and Search](03_trees_and_search.ipynb)) uses.

Binomial coefficients also appear as the coefficients of the **binomial theorem**,

$$(x+y)^n = \sum_{k=0}^{n}\binom{n}{k}x^k y^{n-k},$$

which is why $\binom{n}{k}$ is called a "coefficient" in the first place — expanding $(x+y)^n$
requires choosing, for each of the $n$ factors, whether it contributes an $x$ or a $y$, and
$\binom{n}{k}$ counts the ways to pick exactly $k$ of them to contribute $x$.

### Multinomial coefficients: more than two outcomes

When items get sorted into more than two groups (not just "chosen" vs. "not chosen"), the
**multinomial coefficient** generalizes $\binom{n}{k}$:

$$\binom{n}{k_1,k_2,\dots,k_m} = \frac{n!}{k_1!\,k_2!\,\cdots\,k_m!}, \qquad \sum_{i=1}^m k_i = n,$$

counting the number of ways to partition $n$ distinct items into labeled groups of sizes
$k_1,\dots,k_m$ (or equivalently, the number of distinct arrangements of a sequence with $k_i$
copies of symbol $i$ — the word BANANA has $6!/(3!\,2!\,1!)=60$ distinct letter arrangements,
since it has 3 A's, 2 N's, and 1 B). Setting $m=2$ recovers $\binom{n}{k_1,k_2}=\binom{n}{k_1}$
exactly, so the binomial coefficient is the $m=2$ special case.

### Why growth rate is the whole point

The reason this module opens with counting rather than, say, graphs or trees, is that
**combinatorial explosion** — the fact that $n!$ and $V^L$ (V choices repeated L times) grow
faster than any polynomial — is the recurring obstacle discrete algorithms in AI are built to
avoid. You cannot enumerate every possible sentence, parse, or move sequence once the count
passes a few dozen; every technique in this module (pruning, hashing, dynamic programming,
beam search) exists because exhaustive counting becomes physically impossible long before it
becomes theoretically impossible.

---

In [ ]:
# Setup
import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb, perm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Pascal's triangle makes the recursive identity $\binom{n}{k}=\binom{n-1}{k-1}+\binom{n-1}{k}$
visible directly: every entry is the sum of the two entries above it. Plotting each row's
values also shows *how* binomial coefficients concentrate around the middle as $n$ grows —
the same bell shape behind the normal approximation to the binomial distribution.

In [ ]:
# Pascal's triangle (left) and how binomial coefficients spread out as n grows (right)
N_ROWS = 10
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
row_max = comb(N_ROWS - 1, (N_ROWS - 1) // 2, exact=True)  # normalizes the color scale to the biggest entry drawn
for n in range(N_ROWS):
    for k in range(n + 1):
        val = comb(n, k, exact=True)
        ax.text((N_ROWS - 1) + (k - n / 2) * 1.0, N_ROWS - 1 - n, str(val),
                ha="center", va="center", fontsize=9 - n // 4,
                color=plt.cm.viridis(min(val / row_max, 1.0)))
ax.set_xlim(-1, 2 * N_ROWS)
ax.set_ylim(-1, N_ROWS)
ax.axis("off")
ax.set_title("Pascal's triangle: C(n,k)\neach entry = sum of the two entries above it")

ax2 = axes[1]
for n in [4, 8, 12, 16]:
    ks = np.arange(n + 1)
    vals = [comb(n, k, exact=True) for k in ks]
    ax2.plot(ks, vals, marker="o", ms=4, label=f"n={n}")
ax2.set_xlabel("k"); ax2.set_ylabel("C(n,k)")
ax2.set_title("Binomial coefficients concentrate near the center as n grows")
ax2.legend()
plt.tight_layout()
plt.show()

# Pascal's identity, checked numerically rather than just asserted: every interior entry
# really does equal the sum of the two entries above it.
n, k = 9, 4
lhs = comb(n, k, exact=True)
rhs = comb(n - 1, k - 1, exact=True) + comb(n - 1, k, exact=True)
print(f"Pascal's identity check: C({n},{k})={lhs}, C({n-1},{k-1})+C({n-1},{k})={rhs}, match={lhs == rhs}")

# The binomial theorem: sum of row n of Pascal's triangle is 2^n (set x=y=1 in (x+y)^n)
row_sum = sum(comb(9, k, exact=True) for k in range(10))
print(f"Row 9 of Pascal's triangle sums to {row_sum}, and 2^9 = {2**9} -- matches, as the binomial theorem predicts")

## 🐍 Implementation from Scratch

We implement factorial, permutations, combinations, and the multinomial coefficient from
scratch, and cross-check every one against `scipy.special` and, for small cases, brute-force
enumeration with `itertools` — the ground truth that counting formulas are shortcuts for.

In [ ]:
def factorial(n):
    """n! from scratch -- every other counting formula here is built on this one loop."""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

def n_permutations(n, k):
    """P(n,k): arrangements of k items chosen from n, order matters."""
    return factorial(n) // factorial(n - k)

def n_combinations(n, k):
    """C(n,k): subsets of size k chosen from n, order doesn't matter."""
    return factorial(n) // (factorial(k) * factorial(n - k))

def multinomial(n, group_sizes):
    """Ways to split n distinct items into labeled groups of the given sizes."""
    assert sum(group_sizes) == n, "group sizes must add up to n"
    denom = 1
    for g in group_sizes:
        denom *= factorial(g)
    return factorial(n) // denom

# --- Cross-check against scipy.special (the trusted library implementation) ---
print("n   k   P(n,k) scratch/scipy   C(n,k) scratch/scipy")
for n, k in [(5, 2), (10, 3), (52, 5), (8, 8)]:
    p_scratch, p_scipy = n_permutations(n, k), int(perm(n, k, exact=True))
    c_scratch, c_scipy = n_combinations(n, k), int(comb(n, k, exact=True))
    print(f"{n:<3} {k:<3} {p_scratch:>10}/{p_scipy:<10} {c_scratch:>10}/{c_scipy:<10}")
    assert p_scratch == p_scipy and c_scratch == c_scipy

# --- Cross-check against brute-force enumeration (ground truth for small n) ---
n_small, k_small = 5, 3
brute_perms = len(list(itertools.permutations(range(n_small), k_small)))
brute_combs = len(list(itertools.combinations(range(n_small), k_small)))
print(f"\nn={n_small}, k={k_small}: brute-force count of permutations={brute_perms} "
      f"(formula: {n_permutations(n_small, k_small)}), "
      f"combinations={brute_combs} (formula: {n_combinations(n_small, k_small)})")
assert brute_perms == n_permutations(n_small, k_small)
assert brute_combs == n_combinations(n_small, k_small)

# --- Multinomial coefficient: distinct arrangements of the letters of BANANA ---
word = "BANANA"
from collections import Counter
counts = Counter(word)
n_arrangements = multinomial(len(word), list(counts.values()))
brute_arrangements = len(set(itertools.permutations(word)))
print(f"\nDistinct arrangements of '{word}' (letters: {dict(counts)}): "
      f"formula={n_arrangements}, brute-force (dedup permutations)={brute_arrangements}")
assert n_arrangements == brute_arrangements

# --- 52-choose-5: how many distinct 5-card poker hands exist ---
n_hands = n_combinations(52, 5)
print(f"\nNumber of distinct 5-card poker hands from a 52-card deck: {n_hands:,}")

## 🤖 Why This Matters for AI

An autoregressive language model generates one token at a time, and at every position it could
in principle pick any of $V$ vocabulary tokens. Generating a sequence of length $L$ therefore
has $V^L$ possible completions — exactly the multiplication principle from the Theory section,
applied $L$ times in a row. This number is so large that "find the single best sequence by
checking all of them" is never an option; **beam search** is the practical answer, and the gap
between $V^L$ and what beam search actually explores is a direct, countable illustration of why
combinatorics forces every sequence-generation system to prune.

In [ ]:
# How many candidate sequences would EXHAUSTIVE search have to score, vs. how many
# beam search actually scores? We simulate real beam search (not just the formula) to
# make sure the counted total matches what the algorithm actually touches.

def deterministic_string_seed(s):
    """A reproducible string hash. Python's builtin hash() is randomized per-process
    (PYTHONHASHSEED), so it would silently change every re-run -- this doesn't."""
    h = 0
    for ch in s:
        h = (h * 131 + ord(ch)) % (2**31 - 1)
    return h

def next_log_probs(path, vocab_size, rng_seed=0):
    """Deterministic pseudo-random log-probs for the tokens that could follow `path` --
    stands in for a language model's next-token distribution at this partial sequence."""
    local_rng = np.random.default_rng((deterministic_string_seed(path) ^ rng_seed) % (2**31))
    logits = local_rng.normal(size=vocab_size)
    log_probs = logits - np.log(np.exp(logits).sum())  # normalize to valid log-probabilities
    return log_probs

def beam_search(vocab_size, seq_len, beam_width):
    """Real beam search: at each step, expand every current beam into `vocab_size` children,
    keep only the top `beam_width` by cumulative log-prob, and discard the rest."""
    beams = [("", 0.0)]
    candidates_scored_per_step = []
    for _ in range(seq_len):
        candidates = []
        for path, score in beams:
            log_probs = next_log_probs(path, vocab_size)
            for token in range(vocab_size):
                candidates.append((path + str(token), score + log_probs[token]))
        candidates_scored_per_step.append(len(candidates))  # every one of these gets scored
        candidates.sort(key=lambda pair: -pair[1])
        beams = candidates[:beam_width]
    return beams[0], sum(candidates_scored_per_step)

V = 50  # a small toy vocabulary (real LLMs: 30,000-100,000+ tokens, making this even starker)

print(f"Vocabulary size V={V}. Exhaustive search over all sequences of length L scores V^L paths:")
print(f"{'L':>4} {'V^L (exhaustive)':>22} {'beam k=4 (simulated)':>22} {'exhaustive / beam':>20}")
for L in [5, 10, 20, 40]:
    exhaustive = V ** L
    _, beam_total = beam_search(V, L, beam_width=4)
    print(f"{L:>4} {exhaustive:>22,} {beam_total:>22,} {exhaustive / beam_total:>20,.3g}")

print(f"\nFixing L=20, growing the beam width k instead:")
print(f"{'k':>4} {'beam total scored':>20} {'exhaustive / beam':>20}")
exhaustive_L20 = V ** 20
for k in [1, 2, 4, 8, 16]:
    _, beam_total = beam_search(V, 20, beam_width=k)
    print(f"{k:>4} {beam_total:>20,} {exhaustive_L20 / beam_total:>20,.3g}")

# The simulated totals match a closed-form count exactly: step 1 always scores V candidates
# (from the single empty beam); every later step scores beam_width*V candidates (k beams,
# each expanded V ways). This closed form is itself just the multiplication principle again.
L_check, k_check = 20, 4
closed_form = V * (1 + (L_check - 1) * k_check)
_, simulated_total = beam_search(V, L_check, k_check)
print(f"\nClosed form V*(1+(L-1)*k) for L={L_check}, k={k_check}: {closed_form:,} "
      f"vs simulated: {simulated_total:,} -- match: {closed_form == simulated_total}")

# Combinatorics also governs data augmentation: randomly masking k of n token positions
# (as in masked-language-model pretraining) has C(n,k) distinct masking patterns.
n_positions = 20
print(f"\nMasking patterns for a {n_positions}-token sequence (data augmentation via random masking):")
for k in [1, 2, 5, 10]:
    print(f"  choose {k:>2} positions to mask: C({n_positions},{k}) = {int(comb(n_positions, k, exact=True)):,} distinct patterns")

## 🏋️ Exercises

### Warm-up
1. By hand, compute $\binom{7}{3}$ and $P(7,3)$. Verify both with `n_combinations`/`n_permutations`,
   and confirm $P(7,3) = \binom{7}{3}\times 3!$.

### Practice
2. A tokenizer's vocabulary has 30,000 tokens. Using `n_combinations`, compute how many distinct
   *sets* of 3 tokens could be masked out of a 10-token sequence (i.e. $\binom{10}{3}$ — the
   vocabulary size doesn't matter for this count, only the sequence length does). Then compute
   the multinomial coefficient for splitting a 9-sentence batch into 3 augmentation groups of 3
   sentences each, and confirm it matches `comb(9,3)*comb(6,3)*comb(3,3)`.

### Challenge
3. Extend the beam search simulation to track, at every step, not just the *count* of candidates
   scored but the actual score of the single best sequence found. Run it for beam widths
   $k=1,2,4,8,16$ at $L=20$ and plot best-score-found vs. $k$. Does a bigger beam always find a
   strictly better sequence, and does it ever match the true global optimum you'd get from
   exhaustive search? (You can compute the exhaustive optimum here since $V=50, L=20$ is still
   small enough to reason about the per-step greedy-optimal path — think about when greedy
   per-step choices can and can't be globally optimal.)

---

## 📚 Further Reading
- Graham, Knuth & Patashnik, *Concrete Mathematics*, Ch. 5 (Binomial Coefficients)
- Freitag & Al-Onaizan, ["Beam Search Strategies for Neural Machine Translation"](https://arxiv.org/abs/1702.01806)
- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Section 6, decoding via beam search)

---

*Next notebook: [Graph Theory Fundamentals](02_graph_theory_fundamentals.ipynb)*